# Manning's n Region Polygon Authoring

This workflow writes Manning's n calibration region polygons to the plain-text HEC-RAS geometry file with `GeomLandCover.set_mannings_region_polygons()` and checks the paired regional Manning's n table remains readable.

In [ ]:
from pathlib import Path
import logging
import shutil

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from ras_commander.RasExamples import RasExamples
from ras_commander.geom.GeomLandCover import GeomLandCover
from ras_commander.hdf.HdfLandCover import HdfLandCover

logging.getLogger("ras_commander").setLevel(logging.WARNING)

work_dir = Path("working") / "219_mannings_region_polygon_authoring"
work_dir.mkdir(parents=True, exist_ok=True)

project_path = RasExamples.extract_project(
    "Muncie",
    output_path=work_dir,
    suffix="219_region_polygons",
)

source_geom = project_path / "Muncie.g04"
authoring_geom = project_path / "Muncie_region_polygon_authoring.g04"
shutil.copy2(source_geom, authoring_geom)

source_geom_hdf = project_path / "Muncie.g04.hdf"
regions = HdfLandCover.get_mannings_region_polygons(source_geom_hdf)
flat_area = regions[regions["Name"].astype(str).str.strip() == "Flat Area"].copy()

assert len(flat_area) == 1, "Expected one Muncie 'Flat Area' calibration region"
flat_area[["Name", "geometry"]]


In [ ]:
GeomLandCover.set_mannings_region_polygons(
    authoring_geom,
    flat_area[["Name", "geometry"]],
)

region_n = GeomLandCover.get_region_mannings_n(authoring_geom)
flat_area_n = region_n[region_n["Region Name"] == "Flat Area"].reset_index(drop=True)

assert not flat_area_n.empty, "Region n-value table was not preserved"
assert authoring_geom.with_suffix(".g04.bak").exists(), "Expected a geometry backup"

flat_area_n


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))

flat_area.plot(
    ax=ax,
    facecolor="#9bd3c6",
    edgecolor="#005f73",
    linewidth=2.2,
    alpha=0.42,
)
flat_area.boundary.plot(ax=ax, color="#003f5c", linewidth=2.0)

centroid = flat_area.geometry.iloc[0].centroid
ax.scatter(
    [centroid.x],
    [centroid.y],
    color="#ee9b00",
    edgecolor="#3d2c00",
    s=65,
    zorder=5,
    label="Region centroid",
)
ax.annotate(
    "Flat Area",
    xy=(centroid.x, centroid.y),
    xytext=(18, 16),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    arrowprops={"arrowstyle": "->", "color": "#3d2c00", "lw": 1.2},
)
ax.annotate(
    "N",
    xy=(0.94, 0.91),
    xytext=(0.94, 0.78),
    xycoords="axes fraction",
    textcoords="axes fraction",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    arrowprops={"arrowstyle": "-|>", "color": "black", "lw": 1.4},
)

legend_handles = [
    Patch(
        facecolor="#9bd3c6",
        edgecolor="#005f73",
        alpha=0.42,
        label="Calibration region polygon",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="#ee9b00",
        markeredgecolor="#3d2c00",
        markersize=8,
        label="Region centroid",
    ),
]

ax.set_title("Muncie Manning's n Calibration Region Written to Plain-Text Geometry")
ax.set_xlabel("Project X Coordinate (ft)")
ax.set_ylabel("Project Y Coordinate (ft)")
ax.set_aspect("equal", adjustable="box")
ax.ticklabel_format(style="plain", useOffset=False)
ax.grid(True, color="#d9d9d9", linewidth=0.7)
ax.legend(handles=legend_handles, loc="upper left", frameon=True)

plt.tight_layout()
plt.show()
